In [38]:
import pandas as pd

df=pd.read_csv('transactions.csv')
df.head()

,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD


In [39]:
df.head()

,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD


In [40]:
from sklearn.preprocessing import LabelEncoder
y=df['category']
lf=LabelEncoder()
lf.fit(y)
y=lf.transform(y)
y

array([ 6,  3,  5, ...,  1,  3, 10])

In [41]:
df.head()

,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD


In [42]:
df['category']=y
df.head()

,transaction_description,category,country,currency
0,Wage,6,USA,USD
1,Arby's (Contactless),3,AUSTRALIA,AUD
2,Occupational Therapy,5,USA,USD
3,Potbelly Store Branch,3,UK,GBP
4,Amazon - AUSTRALIA,7,AUSTRALIA,AUD


In [43]:
df['category'].value_counts()

category
9     245047
4     244567
2     244477
6     244366
0     244182
7     243879
8     243281
1     243074
5     243036
3     241392
10         1
Name: count, dtype: int64

In [44]:
currency_encoded=pd.get_dummies(df['currency'], prefix='currency',dtype=int)
currency_encoded.head()


,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,0,0,0,0,1
1,1,0,0,0,0
2,0,0,0,0,1
3,0,0,1,0,0
4,1,0,0,0,0


In [45]:
df.head()  

,transaction_description,category,country,currency
0,Wage,6,USA,USD
1,Arby's (Contactless),3,AUSTRALIA,AUD
2,Occupational Therapy,5,USA,USD
3,Potbelly Store Branch,3,UK,GBP
4,Amazon - AUSTRALIA,7,AUSTRALIA,AUD


In [46]:
df=pd.concat([df, currency_encoded], axis=1)
df=df.drop('currency', axis=1)
df.head()

,transaction_description,category,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,Wage,6,USA,0,0,0,0,1
1,Arby's (Contactless),3,AUSTRALIA,1,0,0,0,0
2,Occupational Therapy,5,USA,0,0,0,0,1
3,Potbelly Store Branch,3,UK,0,0,1,0,0
4,Amazon - AUSTRALIA,7,AUSTRALIA,1,0,0,0,0


In [47]:
import tensorflow as tf
print(tf.__version__)

2.10.0


In [48]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense


In [49]:
unique_words=set()
for transaction in df['transaction_description']:
    for word in transaction.split():
        unique_words.add(word)

len(unique_words)

158067

In [50]:
from collections import Counter

word_freq=Counter()

for transaction in df['transaction_description']:
    words=transaction.lower().split()
    word_freq.update(words)

most_common_10k=word_freq.most_common(10000)

print(most_common_10k[:20])

[('-', 784910), ('store', 375543), ('online', 249633), ('center', 163143), ('branch', 121467), ('india', 97749), ('uk', 97521), ('usa', 97438), ('australia', 97052), ('canada', 96806), ('bank', 73996), ('time', 59779), ('mall', 49805), ('holiday', 35893), ('hospital', 35488), ('hour', 34975), ('company', 32073), ('fitness', 30629), ('business', 30613), ('afternoon', 29986)]


In [51]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [52]:
#NLP preprocessing
df['transaction_description']=df['transaction_description'].str.lower()
import string 
def remove_punctuation(text):
    for ch in string.punctuation:
        text=text.replace(ch,'')
    return text

from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))

def remove_stopwords(text):
    words=text.split()
    list=[]
    for word in words:
        if word not in stop_words:
            list.append(word)
    return ' '.join(list)

df['transaction_description']=df['transaction_description'].apply(remove_punctuation)
df['transaction_description']=df['transaction_description'].apply(remove_stopwords)
df['transaction_description']=df['transaction_description'].apply(str.split)

df.head()

,transaction_description,category,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,[wage],6,USA,0,0,0,0,1
1,"[arbys, contactless]",3,AUSTRALIA,1,0,0,0,0
2,"[occupational, therapy]",5,USA,0,0,0,0,1
3,"[potbelly, store, branch]",3,UK,0,0,1,0,0
4,"[amazon, australia]",7,AUSTRALIA,1,0,0,0,0


In [53]:
def join_words(word_list):
    return ' '.join(word_list)

df['transaction_description']=df['transaction_description'].apply(join_words)
df.head()

,transaction_description,category,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,wage,6,USA,0,0,0,0,1
1,arbys contactless,3,AUSTRALIA,1,0,0,0,0
2,occupational therapy,5,USA,0,0,0,0,1
3,potbelly store branch,3,UK,0,0,1,0,0
4,amazon australia,7,AUSTRALIA,1,0,0,0,0


In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=20000)
xtext=tfidf.fit_transform(df['transaction_description'])
xtext

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6524707 stored elements and shape (2437302, 20000)>

In [55]:
X_currency=df.drop(['transaction_description','category'], axis=1)
X_currency.head()

,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,USA,0,0,0,0,1
1,AUSTRALIA,1,0,0,0,0
2,USA,0,0,0,0,1
3,UK,0,0,1,0,0
4,AUSTRALIA,1,0,0,0,0


In [56]:
y=df['category']
y

0           6
1           3
2           5
3           3
4           7
           ..
2437297     8
2437298     6
2437299     1
2437300     3
2437301    10
Name: category, Length: 2437302, dtype: int32

In [57]:
xtext

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6524707 stored elements and shape (2437302, 20000)>

In [58]:
from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(xtext, y, test_size=0.2, random_state=42)

In [59]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(X_train_text, y_train)
print("Accuracy",accuracy_score(y_test, model.predict(X_test_text)))
print("Classification Report:")
print(classification_report(y_test, model.predict(X_test_text)))

Accuracy 0.9862081274194243
Classification Report:


c:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     49221
           1       1.00      1.00      1.00     48855
           2       1.00      1.00      1.00     48915
           3       1.00      0.99      0.99     47990
           4       0.97      0.99      0.98     48797
           5       0.96      0.98      0.97     48418
           6       0.99      0.99      0.99     48842
           7       0.97      0.94      0.95     48908
           8       0.99      0.99      0.99     48236
           9       0.99      0.99      0.99     49278
          10       0.00      0.00      0.00         1

    accuracy                           0.99    487461
   macro avg       0.90      0.90      0.90    487461
weighted avg       0.99      0.99      0.99    487461



c:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [60]:
df['category'].value_counts()

category
9     245047
4     244567
2     244477
6     244366
0     244182
7     243879
8     243281
1     243074
5     243036
3     241392
10         1
Name: count, dtype: int64

In [61]:
df=df[df['category']!=10]

In [62]:
df.head()

,transaction_description,category,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,wage,6,USA,0,0,0,0,1
1,arbys contactless,3,AUSTRALIA,1,0,0,0,0
2,occupational therapy,5,USA,0,0,0,0,1
3,potbelly store branch,3,UK,0,0,1,0,0
4,amazon australia,7,AUSTRALIA,1,0,0,0,0


In [63]:
X_currency=df.drop(['transaction_description','category'], axis=1)
X_currency.head()

,country,currency_AUD,currency_CAD,currency_GBP,currency_INR,currency_USD
0,USA,0,0,0,0,1
1,AUSTRALIA,1,0,0,0,0
2,USA,0,0,0,0,1
3,UK,0,0,1,0,0
4,AUSTRALIA,1,0,0,0,0


In [64]:
y=df['category']
y

0          6
1          3
2          5
3          3
4          7
          ..
2437296    5
2437297    8
2437298    6
2437299    1
2437300    3
Name: category, Length: 2437301, dtype: int32

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=20000)
xtext=tfidf.fit_transform(df['transaction_description'])
xtext

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6524707 stored elements and shape (2437301, 20000)>

In [66]:
df['category'].value_counts()

category
9    245047
4    244567
2    244477
6    244366
0    244182
7    243879
8    243281
1    243074
5    243036
3    241392
Name: count, dtype: int64

In [67]:
from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(xtext, y, test_size=0.2, random_state=42)

In [68]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(X_train_text, y_train)
print("Accuracy",accuracy_score(y_test, model.predict(X_test_text)))
print("Classification Report:")
print(classification_report(y_test, model.predict(X_test_text)))
print("Confusion Matrix:")
print(confusion_matrix(y_test, model.predict(X_test_text)))

Accuracy 0.986565079052478
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     49292
           1       1.00      1.00      1.00     48727
           2       1.00      1.00      1.00     49017
           3       0.99      0.99      0.99     47880
           4       0.98      0.98      0.98     48752
           5       0.96      0.98      0.97     48350
           6       0.99      1.00      0.99     48804
           7       0.97      0.94      0.95     48875
           8       0.99      0.99      0.99     48491
           9       0.99      0.99      0.99     49273

    accuracy                           0.99    487461
   macro avg       0.99      0.99      0.99    487461
weighted avg       0.99      0.99      0.99    487461

Confusion Matrix:
[[49265     0    27     0     0     0     0     0     0     0]
 [    0 48727     0     0     0     0     0     0     0     0]
 [   67     0 48908     0     0    24     0    

In [69]:
train_accuracy=model.score(X_train_text,y_train)
test_accuracy=model.score(X_test_text,y_test)
print("Train Accuracy =",train_accuracy)
print("Test Accuracy =",test_accuracy)

Train Accuracy = 0.9880379928609527
Test Accuracy = 0.986565079052478


In [70]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,Flatten,TextVectorization
df_small=df.sample(120000,random_state=42)
X_raw_text=df_small['transaction_description']
y_small=df_small['category']
X_train_str,X_test_str,y_train_small,y_test_small=train_test_split(
    X_raw_text,
    y_small,
    test_size=20000,
    random_state=42
)
vectorizer=TextVectorization(
    max_tokens=20000,
    output_mode='int',
    output_sequence_length=15
)
vectorizer.adapt(X_train_str)
X_train_small=np.array(vectorizer(X_train_str))
X_test_small=np.array(vectorizer(X_test_str))
y_train_small=np.array(y_train_small)
y_test_small=np.array(y_test_small)
modelANN=Sequential()
modelANN.add(Embedding(input_dim=20000,output_dim=64,input_length=15))
modelANN.add(Flatten())
modelANN.add(Dense(128,activation='relu'))
modelANN.add(Dense(64,activation='relu'))
modelANN.add(Dense(10,activation='softmax'))
modelANN.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history=modelANN.fit(
    X_train_small,
    y_train_small,
    epochs=5,
    batch_size=512,
    validation_data=(X_test_small,y_test_small),
    verbose=1
)
test_loss,test_accuracy=modelANN.evaluate(X_test_small,y_test_small)
print("Final Train Accuracy =",history.history['accuracy'][-1])
print("Validation Accuracy =",history.history['val_accuracy'][-1])
print("Test Accuracy =",test_accuracy)

Epoch 1/5
196/196 [==============================] - 4s 18ms/step - loss: 0.6452 - accuracy: 0.8717 - val_loss: 0.0331 - val_accuracy: 0.9841
Epoch 2/5
196/196 [==============================] - 3s 18ms/step - loss: 0.0255 - accuracy: 0.9870 - val_loss: 0.0259 - val_accuracy: 0.9833
Epoch 3/5
196/196 [==============================] - 3s 16ms/step - loss: 0.0211 - accuracy: 0.9892 - val_loss: 0.0279 - val_accuracy: 0.9847
Epoch 4/5
196/196 [==============================] - 3s 16ms/step - loss: 0.0171 - accuracy: 0.9908 - val_loss: 0.0278 - val_accuracy: 0.9840
Epoch 5/5
625/625 [==============================] - 1s 2ms/step - loss: 0.0293 - accuracy: 0.9851
Final Train Accuracy = 0.9912999868392944
Validation Accuracy = 0.9851499795913696
Test Accuracy = 0.9851499795913696
